# 04c Team 3, part 2: correction ladder (E11)

**Purpose.** Re-implement Team 3's hybrid NCF + LSTM model and correct its flaws **one at a time**, measuring each step on our protocol (complete validation lists, NDCG@10 with linear gains, exact tie expectation). This shows which of the diagnosed flaws (FINDINGS 5) costs how much: rubric aspect 2, "prior baseline benchmarking and gap diagnosis".

| Rung | Change (cumulative) | Flaw addressed |
|---|---|---|
| **R0** | Team 3 as published: sampled training lists, history filtered by the sampled lists **of all users** (label-dependent, includes validation labels) without the gap to the search, history-vocabulary candidate lookup (attributes misaligned), weighted cross-entropy, ranked by the **maximum class probability** | - |
| **R0b** | the same filter built from **training lists only** | label leak through the catalogue cut |
| **R1** | consistent candidate vocabulary | wrong product lookup |
| **R2** | rank by **expected label** `sum_k k * p_k` | ranking by a confident "no interaction" |
| **R3** | full history, gaps including the gap to the search, corrected time decay, days since the last action | history gutted, time gaps corrupted (and the label leak through filtering) |
| **R4** | train on **complete** candidate lists | downsampled training lists |
| **R5** | add our non-personalised candidate features (G0-G2: relevance, popularity) | no query / candidate-level relevance input |

**How to run on Colab (GPU).**
1. Once: copy the four files of `data/processed/team3/` written by `04b_team3_prep.ipynb` (`train_dev.npz`, `validation.npz`, `lookups.npz`, `meta.json`, about 125 MB) to `MyDrive/BT4222 Group Project/processed/team3/`.
2. In Colab: *File > Open notebook > GitHub*, repository `liamkunert/BT4222-group17-jdsearch`, branch `BRANCH` below, this notebook.
3. *Runtime > Change runtime type > T4 GPU*, then *Run all*. The first cell clones the repository (public) and mounts Drive. All seven rungs take roughly 30-60 minutes on a T4.
4. Results are written to the Drive folder (`ladder_summary.csv`, `ladder_per_search.parquet`). Save the executed notebook (*File > Download > .ipynb*) so the outputs can be committed.

Without a GPU the notebook switches to a **smoke test** (2,000 training and 1,000 validation searches, one epoch) that only checks the pipeline; its numbers are not results.

**Model, as in Team 3's `Models/3.1_Hybrid_NCF_with_LSTM.ipynb`:** item + type + time-bin embeddings (64) into a 2-layer LSTM with attention (empty histories get a learned vector); candidate encoder summing product, brand, shop and L1-L4 embeddings; 15 user meta features through a linear layer; concatenation into an MLP (128, 64, BatchNorm, dropout 0.3) with 4 classes; class-weighted cross-entropy; Adam (lr 1e-3, weight decay 1e-5), ReduceLROnPlateau (factor 0.5, patience 2), batches of 32 searches, at most 10 epochs, best epoch chosen on an inner validation split of the training searches (stopping after 3 epochs without improvement).

In [1]:
# torch first: on Windows, importing numpy / pandas / scikit-learn before torch can break torch's DLL loading.
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence

from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = "feat/baselines"  # branch to clone on Colab; "main" once merged
REPO_ROOT = next((p.resolve() for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/evaluation.py").is_file()), None)
if REPO_ROOT is None and IN_COLAB:  # opened from GitHub: clone the (public) repository
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/liamkunert/BT4222-group17-jdsearch.git", "/content/BT4222-group17-jdsearch"],
                   check=True)
    REPO_ROOT = Path("/content/BT4222-group17-jdsearch")
if REPO_ROOT is None:
    raise FileNotFoundError("Open/clone the repository checkout first.")
if IN_COLAB:
    drive.mount("/content/drive")
    ARTIFACTS = Path("/content/drive/MyDrive/BT4222 Group Project/processed/team3")
else:
    ARTIFACTS = REPO_ROOT / "data/processed/team3"
sys.path.insert(0, str(REPO_ROOT))
from scripts.evaluation import evaluate, paired_difference, summarise

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE = DEVICE.type != "cuda"  # smoke test without a GPU: small subsets, one epoch
MAX_EPOCHS, PATIENCE, BATCH = (1, 1, 32) if SMOKE else (10, 3, 32)
SMOKE_TRAIN, SMOKE_VAL = 2000, 1000
SEEDS = [1]  # add a second seed for R0 and R5 if time allows
info = json.loads((ARTIFACTS / "meta.json").read_text())
print(f"device {DEVICE}; smoke test: {SMOKE}; artifacts {ARTIFACTS}")

Mounted at /content/drive
device cuda; smoke test: False; artifacts /content/drive/MyDrive/BT4222 Group Project/processed/team3


## 1. Load the prepared arrays

In [2]:
def load(split):
    data = dict(np.load(ARTIFACTS / f"{split}.npz"))
    n = len(data["session_id"])
    keep = np.arange(n)
    if SMOKE:
        keep = np.sort(np.random.default_rng(0).choice(n, min(n, SMOKE_TRAIN if split == "train_dev" else SMOKE_VAL),
                                                       replace=False))
    return data, keep


train_data, train_keep = load("train_dev")
val_data, val_keep = load("validation")
lookup = np.load(ARTIFACTS / "lookups.npz")["lookup"]
print(f"training searches {len(train_keep):,}, validation searches {len(val_keep):,}")

training searches 40,000, validation searches 20,000


## 2. Batching

In [3]:
def search_slice(data, i, key):
    off = data[f"{key}_offsets"]
    return slice(off[i], off[i + 1])


def make_batch(data, rows, cfg, train_lists):
    hist_key = f"hist_{cfg['history']}"
    items, types, times, lengths, cands, labels, metas, feats = [], [], [], [], [], [], [], []
    for i in rows:
        h = search_slice(data, i, hist_key)
        items.append(torch.as_tensor(data[f"{hist_key}_item"][h], dtype=torch.long))
        types.append(torch.as_tensor(data[f"{hist_key}_type"][h], dtype=torch.long))
        times.append(torch.as_tensor(data[f"{hist_key}_time"][h], dtype=torch.long))
        lengths.append(h.stop - h.start)
        c = search_slice(data, i, "cand")
        use = np.ones(c.stop - c.start, bool) if train_lists == "full" else data["sampled"][c].astype(bool)
        bug_key = "cand_bug_idx" if cfg["history"] == "filtered" else "cand_bug_idx_train"
        idx = data[bug_key if cfg["lookup"] == "bug" else "cand_idx"][c][use]
        cands.append(torch.as_tensor(idx, dtype=torch.long))
        labels.append(torch.as_tensor(data["label"][c][use], dtype=torch.long))
        metas.append(torch.as_tensor(data[f"meta_{cfg['history']}"][i]))
        feats.append(torch.as_tensor(data["r5_features"][c][use]))
    pad = lambda xs, v: pad_sequence(xs, batch_first=True, padding_value=v)
    empty = torch.zeros(0, dtype=torch.long)
    items = [x if len(x) else empty for x in items]
    return {"items": pad(items, 0) if max(lengths) else torch.zeros(len(rows), 1, dtype=torch.long),
            "types": pad(types, 0) if max(lengths) else torch.zeros(len(rows), 1, dtype=torch.long),
            "times": pad(times, 0) if max(lengths) else torch.zeros(len(rows), 1, dtype=torch.long),
            "lengths": torch.tensor(lengths), "cands": pad(cands, 0), "labels": pad(labels, -1),
            "meta": torch.stack(metas), "feats": pad(feats, 0.0)}

## 3. Team 3's model

In [4]:
class Attention(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.attn = nn.Linear(emb_dim, 1)

    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1).masked_fill(mask == 0, float("-inf"))
        return (x * torch.softmax(scores, dim=1).unsqueeze(-1)).sum(dim=1)


class CandidateEncoder(nn.Module):
    # Sum of product, brand, shop and L1-L4 embeddings; attributes looked up by candidate index.
    def __init__(self, num_wid, num_attr, lookup, emb_dim=64):
        super().__init__()
        self.wid_emb = nn.Embedding(num_wid, emb_dim, padding_idx=0)
        self.attr_emb = nn.ModuleList([nn.Embedding(n, emb_dim, padding_idx=0) for n in num_attr])
        padded = np.zeros((lookup.shape[0], num_wid), np.int64)  # indices beyond the table: unknown attributes
        padded[:, :lookup.shape[1]] = lookup
        self.register_buffer("lookup", torch.as_tensor(padded))

    def forward(self, idx):
        out = self.wid_emb(idx)
        for j, emb in enumerate(self.attr_emb):
            out = out + emb(self.lookup[j][idx])
        return out


class SequentialHybridNCF(nn.Module):
    def __init__(self, num_items, num_types, num_time_bins, num_wid, num_attr, lookup, meta_dim, extra_dim=0,
                 emb_dim=64, hidden_dim=128, num_classes=4):
        super().__init__()
        self.item_emb = nn.Embedding(num_items, emb_dim, padding_idx=0)
        self.type_emb = nn.Embedding(num_types, emb_dim, padding_idx=0)
        self.time_emb = nn.Embedding(num_time_bins, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, emb_dim, batch_first=True, dropout=0.3, num_layers=2)
        self.attention = Attention(emb_dim)
        self.empty_user_emb = nn.Parameter(torch.zeros(1, emb_dim))
        self.candidate_encoder = CandidateEncoder(num_wid, num_attr, lookup, emb_dim)
        self.meta_linear = nn.Linear(meta_dim, emb_dim)
        self.extra_linear = nn.Linear(extra_dim, emb_dim) if extra_dim else None
        fusion_dim = emb_dim * (4 if extra_dim else 3)
        self.mlp = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim), nn.ReLU(), nn.BatchNorm1d(hidden_dim), nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.BatchNorm1d(hidden_dim // 2), nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, num_classes))

    def encode_user(self, items, types, times, lengths):
        x = self.item_emb(items) + self.type_emb(types) + self.time_emb(times)
        out = self.empty_user_emb.repeat(x.size(0), 1)
        has = lengths > 0
        if has.any():
            packed = pack_padded_sequence(x[has], lengths[has].cpu(), batch_first=True, enforce_sorted=False)
            seq, _ = pad_packed_sequence(self.lstm(packed)[0], batch_first=True)
            mask = (items[has][:, :seq.size(1)] != 0) | (torch.arange(seq.size(1), device=seq.device)[None]
                                                         < lengths[has][:, None].to(seq.device))
            out[has] = self.attention(seq, mask)
        return out

    def forward(self, b):
        n = b["cands"].size(1)
        user = self.encode_user(b["items"], b["types"], b["times"], b["lengths"]).unsqueeze(1).expand(-1, n, -1)
        parts = [user, self.candidate_encoder(b["cands"]), self.meta_linear(b["meta"]).unsqueeze(1).expand(-1, n, -1)]
        if self.extra_linear is not None:
            parts.append(self.extra_linear(b["feats"]))
        fusion = torch.cat(parts, dim=-1)
        # As in Team 3: the MLP (with BatchNorm) runs on all positions, padding included.
        return self.mlp(fusion.reshape(-1, fusion.size(-1))).reshape(fusion.size(0), n, -1)

## 4. Training and scoring

In [5]:
def to_device(b):
    return {k: v.to(DEVICE) for k, v in b.items()}


def class_weights(data, rows, train_lists):
    counts = np.zeros(4)
    for i in rows:
        c = search_slice(data, i, "cand")
        labels = data["label"][c] if train_lists == "full" else data["label"][c][data["sampled"][c].astype(bool)]
        counts += np.bincount(labels, minlength=4)
    w = 1.0 / np.maximum(counts, 1)
    return torch.tensor(w / w.sum(), dtype=torch.float32, device=DEVICE)


def run_epoch(model, rows, cfg, criterion, optimizer=None):
    model.train(optimizer is not None)
    total, count = 0.0, 0
    order = np.random.permutation(rows) if optimizer is not None else rows
    for start in range(0, len(order), BATCH):
        b = to_device(make_batch(train_data, order[start:start + BATCH], cfg, cfg["train_lists"]))
        with torch.set_grad_enabled(optimizer is not None):
            logits = model(b)
            mask = b["labels"] != -1
            loss = criterion(logits[mask], b["labels"][mask])
            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total += loss.item() * mask.sum().item()
        count += mask.sum().item()
    return total / count


def train_rung(cfg, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    rows = train_keep
    inner = rows[train_data["inner_val"][rows]]
    fit = rows[~train_data["inner_val"][rows]]
    model = SequentialHybridNCF(
        num_items=info[f"num_hist_{cfg['history']}"], num_types=info["num_types"],
        num_time_bins=info["num_time_bins"],
        num_wid=max(info["num_cand"], info["num_hist_filtered"], info["num_hist_filtered_train"]),
        num_attr=info["num_attr"], lookup=lookup, meta_dim=info["meta_dims"][cfg["history"]],
        extra_dim=len(info["r5_features"]) if cfg["extra"] else 0).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_data, fit, cfg["train_lists"]))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    best, best_state, waited = np.inf, None, 0
    for epoch in range(MAX_EPOCHS):
        start = time.time()
        train_loss = run_epoch(model, fit, cfg, criterion, optimizer)
        val_loss = run_epoch(model, inner, cfg, criterion)
        scheduler.step(val_loss)
        print(f"  epoch {epoch + 1}: train {train_loss:.4f}, inner val {val_loss:.4f}, {time.time() - start:.0f} s")
        if val_loss < best:
            best, waited = val_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def score_validation(model, cfg, lists="full"):
    # Scores for every validation candidate (complete lists, or Team 3's sampled lists).
    model.eval()
    sessions, labels, scores = [], [], []
    for start in range(0, len(val_keep), 64):
        rows = val_keep[start:start + 64]
        b = make_batch(val_data, rows, cfg, lists)
        probs = torch.softmax(model(to_device(b)), dim=-1).cpu()
        s = probs.max(dim=-1).values if cfg["score"] == "maxprob" else (probs * torch.arange(4.0)).sum(-1)
        for j, i in enumerate(rows):
            n = int((b["labels"][j] != -1).sum())
            sessions.append(np.full(n, val_data["session_id"][i]))
            labels.append(b["labels"][j, :n].numpy())
            scores.append(s[j, :n].numpy())
    return pd.DataFrame({"session_id": np.concatenate(sessions), "label": np.concatenate(labels),
                         "score": np.concatenate(scores).astype(float)})

Team 3's own evaluation, verbatim (K = 5 as in their test cell, and K = 10 as reported), for the sampled validation lists.

In [6]:
def team3_metrics(frame, K):
    ndcg, hit = [], []
    for _, g in frame.groupby("session_id", sort=False):
        labels, probs = g.label.to_numpy(), g.score.to_numpy()
        top = np.argsort(-probs)[:min(K, len(probs))]
        top_labels = labels[top]
        hit.append(int((top_labels > 0).any()))
        discounts = np.log2(np.arange(2, len(top_labels) + 2))
        dcg = np.sum((2 ** top_labels - 1) / discounts)
        ideal = np.sort(labels)[::-1][:len(top_labels)]
        idcg = np.sum((2 ** ideal - 1) / discounts) if np.sum(2 ** ideal - 1) > 0 else 1.0
        ndcg.append(dcg / idcg)
    return np.mean(ndcg), np.mean(hit)

## 5. Run the ladder

In [7]:
RUNGS = {
    "R0 Team 3 as published": dict(history="filtered", lookup="bug", score="maxprob", train_lists="sampled", extra=False),
    "R0b - label leak (cut from training lists)": dict(history="filtered_train", lookup="bug", score="maxprob", train_lists="sampled", extra=False),
    "R1 + correct candidate lookup": dict(history="filtered_train", lookup="ok", score="maxprob", train_lists="sampled", extra=False),
    "R2 + rank by expected label": dict(history="filtered_train", lookup="ok", score="expected", train_lists="sampled", extra=False),
    "R3 + full history and time gaps": dict(history="full", lookup="ok", score="expected", train_lists="sampled", extra=False),
    "R4 + train on complete lists": dict(history="full", lookup="ok", score="expected", train_lists="full", extra=False),
    "R5 + relevance and popularity": dict(history="full", lookup="ok", score="expected", train_lists="full", extra=True),
}
val_sizes = pd.Series(np.diff(val_data["cand_offsets"])[val_keep], index=val_data["session_id"][val_keep])
per_search, team3_view, timings = {}, {}, {}
for name, cfg in RUNGS.items():
    for seed in SEEDS:
        key = name if seed == SEEDS[0] else f"{name} (seed {seed})"
        print(key)
        start = time.time()
        model = train_rung(cfg, seed)
        full = score_validation(model, cfg, "full")
        per_search[key] = evaluate(full, "score", expected_sizes=val_sizes)
        sampled = score_validation(model, cfg, "sampled")
        team3_view[key] = {"Team 3 NDCG@5": team3_metrics(sampled, 5)[0], "Team 3 NDCG@10": team3_metrics(sampled, 10)[0],
                           "Team 3 HR@10": team3_metrics(sampled, 10)[1]}
        timings[key] = time.time() - start
        print(f"  NDCG@10 (our protocol) {per_search[key].ndcg.mean():.4f}; {timings[key] / 60:.1f} min")
        del model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

R0 Team 3 as published
  epoch 1: train 1.3520, inner val 1.2935, 116 s
  epoch 2: train 1.2935, inner val 1.2861, 113 s
  epoch 3: train 1.2730, inner val 1.2873, 114 s
  epoch 4: train 1.2413, inner val 1.2975, 113 s
  epoch 5: train 1.1751, inner val 1.3554, 113 s
  NDCG@10 (our protocol) 0.1510; 10.0 min
R0b - label leak (cut from training lists)
  epoch 1: train 1.3529, inner val 1.2936, 111 s
  epoch 2: train 1.2938, inner val 1.2867, 111 s
  epoch 3: train 1.2727, inner val 1.2927, 111 s
  epoch 4: train 1.2426, inner val 1.2997, 111 s
  epoch 5: train 1.1784, inner val 1.3473, 112 s
  NDCG@10 (our protocol) 0.1124; 9.7 min
R1 + correct candidate lookup
  epoch 1: train 1.3780, inner val 1.3297, 112 s
  epoch 2: train 1.3105, inner val 1.3136, 113 s
  epoch 3: train 1.2638, inner val 1.3092, 113 s
  epoch 4: train 1.2033, inner val 1.3308, 113 s
  epoch 5: train 1.1088, inner val 1.4143, 113 s
  epoch 6: train 1.0008, inner val 1.5512, 114 s
  NDCG@10 (our protocol) 0.1558; 11.7

## 6. Results

Reference points on the same validation searches: random order (all candidates tied) and history popularity alone (E09; the `pop_history` column of the R5 inputs, a monotone transform of the count, so the ranking is identical).

In [8]:
reference = {}
frame = pd.DataFrame({"session_id": np.repeat(val_data["session_id"][val_keep], val_sizes.to_numpy())})
c_rows = np.concatenate([np.arange(*val_data["cand_offsets"][[i, i + 1]]) for i in val_keep])
frame["label"] = val_data["label"][c_rows]
frame["score"] = 0.0
reference["random order"] = evaluate(frame, "score", expected_sizes=val_sizes)
frame["score"] = val_data["r5_features"][c_rows, info["r5_features"].index("pop_history")].astype(float)
reference["history popularity (E09)"] = evaluate(frame, "score", expected_sizes=val_sizes)

summary = pd.concat({**{k: summarise(v).iloc[0] for k, v in reference.items()},
                     **{k: summarise(v).iloc[0] for k, v in per_search.items()}}, axis=1).T
summary = summary.join(pd.DataFrame(team3_view).T)
keys = list(per_search)
steps = {keys[0]: paired_difference(per_search[keys[0]], reference["random order"])}
for prev, cur in zip(keys, keys[1:]):
    if "(seed" not in cur and "(seed" not in prev:
        steps[cur] = paired_difference(per_search[cur], per_search[prev])
summary["step_gain"] = pd.Series({k: v[0] for k, v in steps.items()})
summary["step_low"] = pd.Series({k: v[1] for k, v in steps.items()})
summary["step_high"] = pd.Series({k: v[2] for k, v in steps.items()})
summary["minutes"] = pd.Series(timings) / 60
if not SMOKE:
    pd.concat({k: v.ndcg for k, v in {**reference, **per_search}.items()}, axis=1).to_parquet(
        ARTIFACTS / "ladder_per_search.parquet")
    summary.to_csv(ARTIFACTS / "ladder_summary.csv")
summary

,searches,ndcg_at_10,ndcg_ci_low,ndcg_ci_high,recall_at_10,mrr,Team 3 NDCG@5,Team 3 NDCG@10,Team 3 HR@10,step_gain,step_low,step_high,minutes
random order,19772.0,0.084481,0.083040,0.086023,0.159877,0.101972,NaN,NaN,NaN,NaN,NaN,NaN,NaN
history popularity (E09),19772.0,0.340944,0.336386,0.345528,0.526213,0.336507,NaN,NaN,NaN,NaN,NaN,NaN,NaN
R0 Team 3 as published,19772.0,0.151033,0.147667,0.154585,0.237314,0.167346,0.457840,0.518651,0.979719,0.066552,0.063271,0.069803,9.981775
R0b - label leak (cut from training lists),19772.0,0.112380,0.109466,0.115243,0.193310,0.129101,0.464947,0.523803,0.986395,-0.038653,-0.042528,-0.035098,9.651459
R1 + correct candidate lookup,19772.0,0.155752,0.152540,0.159574,0.274039,0.164090,0.543537,0.595444,0.989480,0.043372,0.039752,0.047185,11.730940
R2 + rank by expected label,19772.0,0.140998,0.137614,0.144627,0.248564,0.150371,0.515008,0.569830,0.984422,-0.014754,-0.017965,-0.011540,11.732067
R3 + full history and time gaps,19772.0,0.143060,0.139774,0.146450,0.251254,0.153459,0.518320,0.572967,0.986344,0.002062,-0.000846,0.004562,16.711041
R4 + train on complete lists,19772.0,0.147704,0.144163,0.151133,0.256457,0.158607,0.523007,0.576650,0.986547,0.004643,0.002764,0.006397,16.844682
R5 + relevance and popularity,19772.0,0.296538,0.292254,0.300860,0.492068,0.288286,0.698581,0.734106,0.998129,0.148834,0.144820,0.153161,17.025115


`step_gain`: paired difference in NDCG@10 to the previous rung (R0: to random order), with a 95% bootstrap interval. The Team 3 columns use their own protocol on the sampled validation lists, for comparison with their reported HR@10 0.978 and NDCG@10 0.568.

## Summary (E11, Colab T4, seed 1)

Validation: 19,772 searches with an engaged candidate, complete lists, our protocol. Step = paired difference to the previous rung [95% bootstrap interval over searches; training variance across seeds not included].

| Rung | NDCG@10 | Step | Team 3 protocol NDCG@10 / HR@10 |
|---|---:|---:|---:|
| random order | 0.084 | | 0.532 / 0.992 (E10) |
| history popularity alone (E09) | **0.341** | | 0.742 / 0.998 (E10) |
| R0 Team 3 as published | 0.151 | +0.067 vs random | 0.519 / 0.980 |
| R0b - label leak | 0.112 | -0.039 [-0.043, -0.035] | 0.524 / 0.986 |
| R1 + correct candidate lookup | 0.156 | +0.043 [+0.040, +0.047] | 0.595 / 0.989 |
| R2 + rank by expected label | 0.141 | -0.015 [-0.018, -0.012] | 0.570 / 0.984 |
| R3 + full history and time gaps | 0.143 | +0.002 [-0.001, +0.005] | 0.573 / 0.986 |
| R4 + train on complete lists | 0.148 | +0.005 [+0.003, +0.006] | 0.577 / 0.987 |
| R5 + relevance and popularity | 0.297 | +0.149 [+0.145, +0.153] | 0.734 / 0.998 |

- **The rebuild matches Team 3's reported behaviour on their protocol** (R0: HR@10 0.980 vs their 0.978; NDCG@10 0.519 vs their 0.568, which their code computed with K = 5 and a truncated ideal DCG), and on that protocol it scores **below random order** (0.532).
- **More than half of R0's gain over random is the label leak** through their catalogue cut: 0.039 of 0.067.
- **The code and design fixes add little** on their architecture: correct lookup +0.043; complete lists +0.005; full history and time gaps +0.002 (not significant); ranking by expected label -0.015 (hypothesis, untested: inverse-frequency class weights inflate cart and purchase probabilities, so the expected label is noisy; R1 and R2 are also separate training runs). All corrections together (R0b to R4): +0.035 [+0.032, +0.039].
- **Adding relevance and popularity helps most (+0.149) but the model still ranks below sorting by popularity alone:** R5 0.297 vs 0.341, difference -0.044 [-0.048, -0.041]. A per-candidate classifier over summed ID embeddings, trained with cross-entropy, does not exploit even the strongest single signal.
- All rungs overfit early (best epoch 2-3 of at most 10; stopped after 5-6), as in Team 3's own runs.
- **Consequence:** Team 3's architecture is not a viable base. The comparison our models must win is against popularity and a trained non-personalised ranker, not against Team 3.
